# Datetime operations
## Refreshing basic datetime operations

In [ ]:
import pandas as pd

df = pd.DataFrame({
    "event": ["Orientation", "Lecture", "Exam", "Workshop", "Holiday"],
    "date": ["2024-08-20", "2024-08-21", "2024-8-22", "2024-08-24", "invalid_date"]
})

print(df)


### Convert to datetime date type

In [ ]:
df["date"] = pd.to_datetime(df["date"], errors="coerce")

print(df)


### Use `.dt` Accessor and create new columns

In [ ]:
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month
df["day"] = df["date"].dt.day
df["weekday"] = df["date"].dt.weekday  # Monday=0, Sunday=6
df['is_weekend'] = df['date'].dt.weekday >=5

print(df)


## Period date type
A Period represents a span of time, not a single point.
### Create Monthly Periods

In [ ]:
import pandas as pd

# Create a monthly period
p = pd.Period("2024-08", freq="M")

print(p)
print(type(p))


### Access attributes

In [ ]:
print(p.year)
print(p.month)
print(p.start_time)
print(p.end_time)


### Period arithmetic

In [ ]:
print(p + 1)   # next month
print(p - 2)   # two months earlier
print((p+1).start_time)


### Create period Series
Convert datetime to period Series, you can use using `.dt.to_period("M")`

In [ ]:
df = pd.DataFrame({
    "month": ["2024-06", "2024-07", "2024-08"]
})

df["month"] = pd.to_datetime(df["month"]).dt.to_period("M")

print(df)


### Group by Period (time aggregation)

In [ ]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    "date": pd.date_range(start="2024-08-01", periods=10, freq="3B"), # "3B" means three business days
    "sales": [10, 15, 12, 20, 18, 25, 30, 28, 22, 19]
})

df


In [ ]:
# convert to monthly period

df["month"] = df["date"].dt.to_period("M")

df


In [ ]:
# group by month

monthly_sales = df.groupby("month")["sales"].sum()

monthly_sales


## Filtering by datetime

example: create class schedule for EST 389


In [ ]:
import pandas as pd

# Create date range (daily first)
dates = pd.date_range(start="2026-01-26", end="2026-05-06", freq="D")

# Remove spring break
dates = dates[~dates.to_series().between("2026-03-16", "2026-03-22")]

# Filter Mondays (0) and Wednesdays (2)
mw_dates = dates[dates.weekday.isin([0, 2])]

# Add time (9:30 AM)
mw_start_times = mw_dates + pd.Timedelta(hours=9, minutes=30)
mw_end_times = mw_start_times + pd.Timedelta(hours=1, minutes=20)

# Create DataFrame
df = pd.DataFrame({"class_time": mw_dates,
                   "start_time": mw_start_times,
                   "end_time": mw_end_times})

df["class_period"] = pd.IntervalIndex.from_arrays(df["start_time"], df["end_time"], closed="both")
df['week'] = df['class_time'].dt.to_period("W").rank(method='dense').astype(int)

df.head()


In [ ]:
# Filter a Specific Time Period (March)

march_classes = df[
    (df["class_time"] >= "2026-03-01") &
    (df["class_time"] <= "2026-03-31")
]

march_classes

In [ ]:
# Alternative way to filter March class

df[df["class_time"].dt.month == 3]


In [ ]:
# Filter out Mondays, the days where we will have "Data Science Project of the Day"

mondays = df[df["class_time"].dt.weekday == 0]

mondays.head()


In [ ]:
# Create a new column to indicate whether we will have data science project of the day
df['have_ds_project'] = df["class_time"].dt.weekday == 0

df

## Compute time differences and sort by time

In [ ]:
import pandas as pd

df = pd.DataFrame({
    "class": ["EST389", "EST532", "CSE232", "AMP314"],
    "start_time": [
        "2026-02-09 09:30",
        "2026-02-02 09:30",
        "2026-02-11 09:30",
        "2026-02-04 09:30"
    ],
    "end_time": [
        "2026-02-09 10:40",
        "2026-02-02 10:45",
        "2026-02-11 11:00",
        "2026-02-04 10:50"
    ]
})

# Convert to datetime
df["start_time"] = pd.to_datetime(df["start_time"])
df["end_time"] = pd.to_datetime(df["end_time"])

df


In [ ]:
# compute duration

df["duration"] = df["end_time"] - df["start_time"]

df


In [ ]:
# convert durations to minutes

df["duration_min"] = df["duration"].dt.total_seconds() / 60

df


In [ ]:
# Compute Time Between Classes

df["gap_since_last"] = df["start_time"].diff()

df


In [ ]:
# See that gap_between_last has negative numbers, so it is better we sort by time first

df = df.sort_values(by="start_time", ascending=True)

df["gap_since_last"] = df["start_time"].diff()

df


In [ ]:
# group by the day of the week

df['weekday'] = df['start_time'].dt.day_name()

df.groupby("weekday")['class'].count().reset_index().rename(columns={'class': 'class_count'})
